# Practical 1 — Ratio analyser (tutor fallback notebook)

This is a model answer of the kind a well-specified prompt should produce. Use it if a pair's AI output goes badly wrong, or to show the class a reference version after the reconciliation step.

**Run:** Runtime → Run all, then upload `ratios_data.csv` when asked. Figures are in £000.

In [ ]:
# Upload the CSV when running in Google Colab; otherwise read it from the current folder.
import pandas as pd
try:
    from google.colab import files
    uploaded = files.upload()          # choose ratios_data.csv when prompted
except ImportError:
    pass

df = pd.read_csv('ratios_data.csv')
df

## 1. Check the data before calculating anything
A good tool reports missing, zero and negative values first, so the accountant decides how to treat them.

In [ ]:
checks = []
for _, r in df.iterrows():
    tag = f"{r.company} {r.year}"
    if pd.isna(r.inventory):
        checks.append(f"{tag}: inventory is MISSING (blank is not the same as zero)")
    if r.current_liabilities == 0:
        checks.append(f"{tag}: current liabilities are ZERO - likely a data entry error")
    if r.equity <= 0:
        checks.append(f"{tag}: equity is NEGATIVE ({r.equity:,.0f}) - debt-to-equity is not meaningful")
print("\n".join(checks) if checks else "No data problems found")

## 2. Calculate the ratios
Each ratio is only calculated when its inputs are valid; otherwise the cell is left blank (NaN) and a warning is printed above.

In [ ]:
import numpy as np

def safe_div(num, den, valid):
    return np.where(valid, num / den.replace(0, np.nan), np.nan)

cl_ok  = df.current_liabilities > 0
inv_ok = df.inventory.notna()
eq_ok  = df.equity > 0

out = df[["company", "year"]].copy()
out["current_ratio"]    = safe_div(df.current_assets, df.current_liabilities, cl_ok)
out["quick_ratio"]      = safe_div(df.current_assets - df.inventory, df.current_liabilities, cl_ok & inv_ok)
out["gross_margin"]     = (df.revenue - df.cost_of_sales) / df.revenue
out["operating_margin"] = df.operating_profit / df.revenue
out["debt_to_equity"]   = safe_div(df.total_debt, df.equity, eq_ok)
out.round(2)

## 3. Reconcile with the hand calculation
Brindley Engineering Ltd 2023 should give: current 1.60, quick 0.90, gross margin 30%, operating margin 10%, D/E 0.50.

In [ ]:
row = out[(out.company == "Brindley Engineering Ltd") & (out.year == 2023)].iloc[0]
hand = {"current_ratio": 1.60, "quick_ratio": 0.90, "gross_margin": 0.30, "operating_margin": 0.10, "debt_to_equity": 0.50}
for k, v in hand.items():
    print(f"{k:17s} tool {row[k]:.2f}   hand {v:.2f}   {'OK' if abs(row[k]-v) < 0.005 else 'DIFFERENCE'}")

## 4. Extension — margins over time

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for name, g in out.groupby("company"):
    ax.plot(g.year, g.operating_margin * 100, marker="o", label=name)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(sorted(out.year.unique()))
ax.set_ylabel("Operating margin (%)")
ax.set_title("Operating margin by company")
ax.legend(frameon=False)
plt.show()

## Discussion: what a naive tool gets wrong
- **Corve 2024:** treating blank inventory as 0 gives a quick ratio of 1.07, the same as the current ratio. It looks plausible, which is the danger.
- **Brindley 2025:** dividing by zero current liabilities returns `inf` or crashes.
- **Wrekin 2025:** D/E of -5.21 looks like very low gearing if nobody reads the sign.
- **Wrekin 2023–25:** zero inventory is genuine for a software company, so the quick ratio equals the current ratio. Zero is not missing.